# Case Study 2: AI Customer Support Agent for Banking

This notebook implements a secure AI-powered banking customer support agent using LangChain.
The agent handles common banking queries (balance checks, transaction lookups, card management)
with built-in risk classification and identity verification.

**Key principles:**
- Never expose full account numbers
- Classify every query by risk level before acting
- Require identity verification for medium-risk operations
- Escalate high-risk scenarios to human agents

In [1]:
import os
import json
from datetime import datetime, timedelta
from typing import Optional

from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import HumanMessage, AIMessage

import os
os.environ["GOOGLE_API_KEY"] = "your-gemini-api-key-here"

print("Dependencies loaded.")

Dependencies loaded.


In [2]:
# Mock Customer Data
CUSTOMERS = {
    "C001": {
        "account_id": "9382710045", "name": "Priya Sharma", "balance": 3247.50,
        "phone_last4": "1234", "security_answer": "mumbai",
        "card_status": "active", "card_last4": "5678",
        "recent_transactions": [
            {"id": "TXN001", "merchant": "Amazon", "amount": 47.99, "date": "2026-06-23", "status": "pending"},
            {"id": "TXN002", "merchant": "Starbucks", "amount": 5.75, "date": "2026-06-22", "status": "completed"},
            {"id": "TXN003", "merchant": "Netflix", "amount": 15.99, "date": "2026-06-20", "status": "completed"},
            {"id": "TXN004", "merchant": "Shell Gas", "amount": 42.30, "date": "2026-06-19", "status": "completed"},
        ],
    },
    "C002": {
        "account_id": "7461029983", "name": "James Wilson", "balance": 12580.00,
        "phone_last4": "9876", "security_answer": "golden",
        "card_status": "blocked", "card_last4": "4321",
        "recent_transactions": [
            {"id": "TXN010", "merchant": "ElectroMart - Miami, FL", "amount": 500.00, "date": "2026-06-24", "status": "completed"},
            {"id": "TXN011", "merchant": "Uber", "amount": 23.40, "date": "2026-06-23", "status": "completed"},
            {"id": "TXN012", "merchant": "Target", "amount": 156.80, "date": "2026-06-21", "status": "completed"},
        ],
    },
    "C003": {
        "account_id": "5520184736", "name": "Maria Garcia", "balance": 890.25,
        "phone_last4": "5555", "security_answer": "bluejay",
        "card_status": "active", "card_last4": "8899",
        "recent_transactions": [
            {"id": "TXN020", "merchant": "Walmart", "amount": 234.56, "date": "2026-06-24", "status": "pending"},
            {"id": "TXN021", "merchant": "CVS Pharmacy", "amount": 18.99, "date": "2026-06-22", "status": "completed"},
        ],
    },
}

verified_sessions: dict[str, bool] = {}

def mask_account_id(account_id: str) -> str:
    return "X" * (len(account_id) - 4) + account_id[-4:]

def get_customer(customer_id: str) -> Optional[dict]:
    return CUSTOMERS.get(customer_id)

print(f"Loaded {len(CUSTOMERS)} mock customer accounts.")

Loaded 3 mock customer accounts.


In [3]:
# Risk Classification
RISK_LEVELS = {
    "check_balance": "LOW", "transaction_history": "LOW", "pending_transaction": "LOW",
    "card_status": "LOW", "general_inquiry": "LOW",
    "card_unblock": "MEDIUM", "dispute_transaction": "MEDIUM", "update_contact": "MEDIUM",
    "large_transfer": "HIGH", "account_closure": "HIGH", "suspicious_activity": "HIGH",
}

def classify_risk(intent: str) -> str:
    return RISK_LEVELS.get(intent, "HIGH")

for intent in ["check_balance", "card_unblock", "suspicious_activity", "unknown"]:
    print(f"{intent:25s} -> {classify_risk(intent)}")

check_balance             -> LOW
card_unblock              -> MEDIUM
suspicious_activity       -> HIGH
unknown                   -> HIGH


In [4]:
# Identity Verification Simulator
def verify_identity(customer_id: str, provided_info: dict) -> dict:
    customer = get_customer(customer_id)
    if not customer:
        return {"verified": False, "message": "Customer not found."}
    if "phone_last4" in provided_info:
        if provided_info["phone_last4"] != customer["phone_last4"]:
            return {"verified": False, "message": "Phone number does not match our records."}
    if "otp" in provided_info:
        if provided_info["otp"] != "123456":
            return {"verified": False, "message": "Invalid OTP. Please try again."}
    if provided_info:
        verified_sessions[customer_id] = True
        return {"verified": True, "message": f"Identity verified for {customer['name']}."}
    return {"verified": False, "message": "No verification information provided."}

print(verify_identity("C001", {"phone_last4": "1234", "otp": "123456"}))
print(verify_identity("C001", {"phone_last4": "0000"}))

{'verified': True, 'message': 'Identity verified for Priya Sharma.'}
{'verified': False, 'message': 'Phone number does not match our records.'}


In [5]:
# Banking Tools
@tool
def check_balance(account_id: str) -> str:
    """Check the balance for a bank account. Returns masked account number and balance."""
    for cust in CUSTOMERS.values():
        if cust["account_id"] == account_id:
            masked = mask_account_id(account_id)
            pending = sum(t["amount"] for t in cust["recent_transactions"] if t["status"] == "pending")
            return f"Account {masked}:\n  Available Balance: ${cust['balance'] - pending:,.2f}\n  Current Balance: ${cust['balance']:,.2f}\n  Pending Holds: ${pending:,.2f}"
    return "Account not found."

@tool
def get_transactions(account_id: str) -> str:
    """Get recent transactions for a bank account."""
    for cust in CUSTOMERS.values():
        if cust["account_id"] == account_id:
            lines = [f"Recent transactions for account {mask_account_id(account_id)}:\n"]
            for t in cust["recent_transactions"]:
                lines.append(f"  {t['date']} | {t['merchant']:25s} | ${t['amount']:>8.2f} | {t['status']}")
            return "\n".join(lines)
    return "Account not found."

@tool
def check_card_status(account_id: str) -> str:
    """Check the debit card status for a bank account."""
    for cust in CUSTOMERS.values():
        if cust["account_id"] == account_id:
            return f"Card ending in {cust['card_last4']}: {cust['card_status'].upper()}\nAccount holder: {cust['name']}"
    return "Account not found."

@tool
def request_card_unblock(account_id: str, customer_id: str) -> str:
    """Request to unblock a debit card. Requires prior identity verification."""
    if not verified_sessions.get(customer_id, False):
        return "BLOCKED: Identity verification required before unblocking. Please verify identity first."
    for cid, cust in CUSTOMERS.items():
        if cust["account_id"] == account_id and cid == customer_id:
            if cust["card_status"] == "active":
                return f"Card ending in {cust['card_last4']} is already active."
            cust["card_status"] = "active"
            return f"Card ending in {cust['card_last4']} has been unblocked. It will be active within 15 minutes."
    return "Account not found or customer ID mismatch."

@tool
def verify_customer_identity(customer_id: str, phone_last4: str, otp: str) -> str:
    """Verify a customer's identity using phone number (last 4 digits) and OTP code."""
    result = verify_identity(customer_id, {"phone_last4": phone_last4, "otp": otp})
    return result["message"]

@tool
def escalate_to_human(customer_id: str, reason: str, conversation_summary: str) -> str:
    """Escalate a case to a human agent for high-risk scenarios."""
    case_number = f"FR-2026-{hash(customer_id + reason) % 100000:05d}"
    return f"Case {case_number} created and escalated.\nReason: {reason}\nEstimated wait: under 2 minutes."

tools = [check_balance, get_transactions, check_card_status, request_card_unblock, verify_customer_identity, escalate_to_human]
print(f"Registered {len(tools)} banking tools.")

Registered 6 banking tools.


In [6]:
# Create the Banking Agent

BANKING_SYSTEM_PROMPT = """You are a secure AI customer support agent for SecureBank.

CRITICAL SECURITY RULES:
1. NEVER reveal full account numbers. Always use masked format (e.g., XXXXXX4321).
2. NEVER share one customer's data with another customer.
3. For MEDIUM-risk operations (card unblock, disputes), ALWAYS verify identity first.
4. For HIGH-risk scenarios (suspicious activity, large transfers, account closure),
   gather context then escalate to a human agent immediately.
5. Be empathetic but professional.

RISK CLASSIFICATION:
- LOW (auto-resolve): balance checks, transaction history, card status
- MEDIUM (verify then act): card unblock, dispute filing
- HIGH (escalate): suspicious activity, large transfers, account closure

CURRENT CUSTOMER: C001 (Account XXXXXX0045)

Always end interactions by asking if there's anything else you can help with."""

agent = create_agent(
    "google_genai:gemini-2.5-flash",
    tools=tools,
    system_prompt=BANKING_SYSTEM_PROMPT,
)

print("Banking agent created.")

Banking agent created.


In [7]:
# Demo Conversations

def _clean(content):
    if isinstance(content, list):
        return "".join(b["text"] for b in content if isinstance(b, dict) and "text" in b)
    return str(content) if content else ""

def run_demo(messages: list[str], title: str):
    print(f"\n{'='*70}")
    print(f"  DEMO: {title}")
    print(f"{'='*70}")
    for msg in messages:
        print(f"\n>> Customer: {msg}")
        result = agent.invoke({"messages": [HumanMessage(content=msg)]})
        response = _clean(result["messages"][-1].content)
        print(f"\n<< Agent: {response}")

run_demo(["Why is my card transaction at Amazon still pending?"],
         title="Pending Transaction Inquiry (LOW risk)")

run_demo(["My debit card is blocked. I need it unblocked."],
         title="Blocked Card (MEDIUM risk)")

run_demo(["What\'s my account balance?"],
         title="Balance Check (LOW risk)")

run_demo(["I see a suspicious $500 charge from a store I\'ve never visited."],
         title="Suspicious Activity (HIGH risk — Escalation)")


  DEMO: Pending Transaction Inquiry (LOW risk)

>> Customer: Why is my card transaction at Amazon still pending?

<< Agent: I apologize, but I am unable to retrieve the transaction details for you at this moment. It seems there might be an issue accessing your account information.

I will escalate this to a human agent who can investigate why your Amazon transaction is still pending and help you further. Please provide a brief summary of our conversation for the agent.

  DEMO: Blocked Card (MEDIUM risk)

>> Customer: My debit card is blocked. I need it unblocked.

<< Agent: I can help you with that. To unblock your debit card, I need to verify your identity first. Please provide the last 4 digits of your phone number and the One-Time Password (OTP) sent to your registered mobile number.

  DEMO: Balance Check (LOW risk)

>> Customer: What's my account balance?

<< Agent: I apologize, but I was unable to retrieve your account balance. The system indicates that the account XXXXXX0045 w